## 1. Load the data

In [3]:
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

# utf-8-sig removes the byte-order mark some exports put before the first column name
df = pd.read_csv("data/tayseer_services.csv", parse_dates=["month"], encoding="utf-8-sig")
df.head()

,month,region,service_category,channel,transactions,unique_users,digital_adoption_pct,avg_completion_min,cost_per_txn_sar,csat,sla_met_pct,complaints
0,2021-07-01,Al-Baha,Business & Licensing,Branch,5727,4745,33.4,117.0,51.88,3.05,80.7,10
1,2021-07-01,Al-Baha,Business & Licensing,Call Center,2575,1708,33.4,31.5,24.11,3.57,93.2,2
2,2021-07-01,Al-Baha,Business & Licensing,Mobile App,1936,721,33.4,13.7,6.26,4.13,96.2,1
3,2021-07-01,Al-Baha,Business & Licensing,Web Portal,2425,1221,33.4,19.4,8.63,3.85,94.9,3
4,2021-07-01,Al-Baha,Education & Training,Branch,7443,5923,37.0,29.4,39.00,3.55,84.4,7


## 2. Check the data

In [4]:
print("Rows, columns:", df.shape)
print("Missing values:", df.isna().sum().sum())
print("Duplicate keys:", df.duplicated(["month", "region", "service_category", "channel"]).sum())
print()
print(df.dtypes)

Rows, columns: (28080, 12)
Missing values: 0
Duplicate keys: 0

month                   datetime64[us]
region                             str
service_category                   str
channel                            str
transactions                     int64
unique_users                     int64
digital_adoption_pct           float64
avg_completion_min             float64
cost_per_txn_sar               float64
csat                           float64
sla_met_pct                    float64
complaints                       int64
dtype: object


**Check:** 28,080 rows × 12 columns, no missing values, no duplicate keys, and `month` is a real date, so no cleaning is needed.

## 3. Dataset scope

In [5]:
print("From", df["month"].min().date(), "to", df["month"].max().date(),
      "|", df["month"].nunique(), "months")
print()
for col in ["region", "service_category", "channel"]:
    print(col, "→", df[col].nunique(), "values")
    print(sorted(df[col].unique()))
    print()

print("13 × 9 × 4 × 60 =", 13 * 9 * 4 * 60, "= the row count, so every combination is present.")

From 2021-07-01 to 2026-06-01 | 60 months

region → 13 values
['Al-Baha', 'Al-Jouf', 'Asir', 'Eastern Province', 'Hail', 'Jazan', 'Madinah', 'Makkah', 'Najran', 'Northern Borders', 'Qassim', 'Riyadh', 'Tabuk']

service_category → 9 values
['Business & Licensing', 'Education & Training', 'Health Services', 'Identity & Civil Affairs', 'Justice & Notary', 'Labour & HR Services', 'Municipal Services', 'Social Support & Subsidies', 'Traffic & Transport']

channel → 4 values
['Branch', 'Call Center', 'Mobile App', 'Web Portal']

13 × 9 × 4 × 60 = 28080 = the row count, so every combination is present.


## 4. The repeated adoption values

The brief warns that `digital_adoption_pct` repeats across the four channels of each
month × region × category. The cell below **proves** this on the whole dataset before relying on it,
rather than assuming it. If any cell had different values per channel, de-duplicating would
silently drop real data.

In [6]:
# One example cell, all four channels
sample = df[(df["month"] == "2021-07-01") &
            (df["region"] == "Al-Baha") &
            (df["service_category"] == "Business & Licensing")]
print(sample[["channel", "transactions", "digital_adoption_pct"]].to_string(index=False))

# The same test across all 7,020 cells
per_cell = df.groupby(["month", "region", "service_category"])["digital_adoption_pct"].nunique()
print()
print("Cells in total:", len(per_cell))
print("Cells where adoption differs between channels:", (per_cell > 1).sum())
print()
print("Sum of the raw column (meaningless — counts each value 4 times):",
      f"{df['digital_adoption_pct'].sum():,.0f}")

    channel  transactions  digital_adoption_pct
     Branch          5727                  33.4
Call Center          2575                  33.4
 Mobile App          1936                  33.4
 Web Portal          2425                  33.4

Cells in total: 7020
Cells where adoption differs between channels: 0

Sum of the raw column (meaningless — counts each value 4 times): 1,413,145


**Result:** adoption is identical across all four channels in every one of the 7,020 cells.
It is therefore an attribute **of the cell**, not of the channel, and must be counted once.
Summing the raw column would count every value four times.

## 5. Metric and aggregation

- **Adoption per cell:** one value per month × region × category (`drop_duplicates`), never summed.
- **Adoption per region:** a **transaction-weighted mean** of its 9 category cells, so a category handling 200,000 transactions counts more than one handling 2,000:

$$
\text{adoption} =
\frac{\sum(\text{adoption}_{cell} \times \text{transactions}_{cell})}
{\sum \text{transactions}_{cell}}
$$

- **Transactions** are counts, so they can be summed across the four channels.
- **Change** is reported in **percentage points (pp)** — the arithmetic difference between two percentages, not a percent change.

In [7]:
# One adoption value per month x region x category
adoption = df[["month", "region", "service_category", "digital_adoption_pct"]].drop_duplicates()

# Transactions summed over the four channels of each cell
transactions = (df.groupby(["month", "region", "service_category"], as_index=False)["transactions"]
                  .sum())

cell = adoption.merge(transactions, on=["month", "region", "service_category"])
cell["weighted"] = cell["digital_adoption_pct"] * cell["transactions"]

print("Cells:", len(cell), "(expected 13 × 9 × 60 =", 13 * 9 * 60, ")")

# Region-level and national adoption, by month
region = (cell.groupby(["month", "region"])["weighted"].sum()
          / cell.groupby(["month", "region"])["transactions"].sum()).unstack()
national = (cell.groupby("month")["weighted"].sum()
            / cell.groupby("month")["transactions"].sum())

print(f"National adoption: {national['2021-07-01']:.1f}% (Jul 2021) → "
      f"{national['2024-06-01']:.1f}% (Jun 2024) → {national['2026-06-01']:.1f}% (Jun 2026)")

Cells: 7020 (expected 13 × 9 × 60 = 7020 )
National adoption: 34.2% (Jul 2021) → 56.7% (Jun 2024) → 64.0% (Jun 2026)


## 6. Growth per region, and the rule for "slowed"

A region was classified as *slowed* when its 24-month gain was less than half the national gain (< 3.62 pp). The regions were identified using this rule in code.

In [8]:
START, MID, LAST = "2021-07-01", "2024-06-01", "2026-06-01"
PREV = "2025-06-01"

nat_growth = national[LAST] - national[MID]
THRESHOLD = nat_growth / 2

gain = region.loc[LAST] - region.loc[MID]

compare = pd.DataFrame({
    "Jun 2024": region.loc[MID],
    "Jun 2026": region.loc[LAST],
    "2-year gain (pp)": gain,
    "1-year gain (pp)": region.loc[LAST] - region.loc[PREV],
    "First 3 years (pp)": region.loc[MID] - region.loc[START],
}).sort_values("2-year gain (pp)")

compare["slowed"] = gain < THRESHOLD
SLOWED = compare.index[compare["slowed"]].tolist()

print(f"National 2-year gain: {nat_growth:.2f} pp  →  threshold = {THRESHOLD:.2f} pp")
print("Regions below the threshold:", SLOWED)
compare.round(2)

National 2-year gain: 7.23 pp  →  threshold = 3.62 pp
Regions below the threshold: ['Al-Baha', 'Najran', 'Northern Borders', 'Asir', 'Jazan']


,Jun 2024,Jun 2026,2-year gain (pp),1-year gain (pp),First 3 years (pp),slowed
region,,,,,,
Al-Baha,53.31,55.52,2.21,0.71,17.18,True
Najran,50.12,52.37,2.25,0.57,18.11,True
Northern Borders,59.42,61.67,2.25,0.84,17.75,True
Asir,57.89,60.16,2.27,0.81,18.62,True
Jazan,47.43,50.04,2.61,0.89,19.09,True
Riyadh,60.04,67.78,7.74,2.27,23.49,False
Eastern Province,59.54,67.37,7.83,2.27,23.20,False
Tabuk,57.24,65.10,7.86,2.12,23.92,False
Makkah,55.20,63.11,7.91,2.26,23.10,False


**Reading the table.** The five regions gained **2.2–2.6 pp** in two years; the other eight gained
**7.7–8.5 pp**. Nothing falls between 2.6 and 7.7, so this is a clear separation rather than a gradual
spread. The **first 3 years** column shows that these five regions gained **17–19 pp** before mid-2024,
so the slowdown represents a change in their recent trajectory rather than a consistently low-growth pattern.

## 7. Robustness checks

Three questions that could overturn the finding, tested before it is reported.

**(a) Does the choice of weighting change the answer?**

In [9]:
users = df.groupby(["month", "region", "service_category"], as_index=False)["unique_users"].sum()
c2 = cell.merge(users, on=["month", "region", "service_category"])

def growth_using(weight_col):
    w = c2["digital_adoption_pct"] * c2[weight_col]
    r = (w.groupby([c2["month"], c2["region"]]).sum()
         / c2[weight_col].groupby([c2["month"], c2["region"]]).sum()).unstack()
    return r.loc[LAST] - r.loc[MID]

c2["one"] = 1.0
robust = pd.DataFrame({
    "by transactions": growth_using("transactions"),
    "by unique users": growth_using("unique_users"),
    "unweighted mean": growth_using("one"),
}).sort_values("by transactions")
robust.round(2)

,by transactions,by unique users,unweighted mean
region,,,
Al-Baha,2.21,2.21,2.23
Najran,2.25,2.23,2.18
Northern Borders,2.25,2.28,2.26
Asir,2.27,2.29,2.30
Jazan,2.61,2.62,2.64
Riyadh,7.74,7.75,7.73
Eastern Province,7.83,7.84,7.78
Tabuk,7.86,7.87,7.89
Makkah,7.91,7.92,7.87


**Result:** the three methods agree to within **0.1 percentage points per region**. The finding does not depend on how the regional average is calculated.

**(b) Is the slowdown limited to one or two service categories?**

In [10]:
cat = cell.pivot_table(index=["region", "service_category"],
                       columns="month", values="digital_adoption_pct")
cat_growth = (cat[LAST] - cat[MID]).unstack("service_category")

print("Largest 2-year gain inside the five slowed regions:",
      round(cat_growth.loc[SLOWED].max().max(), 2), "pp")
print("Smallest 2-year gain anywhere else:              ",
      round(cat_growth.drop(SLOWED).min().min(), 2), "pp")
cat_growth.round(1)

Largest 2-year gain inside the five slowed regions: 3.2 pp
Smallest 2-year gain anywhere else:               7.0 pp


service_category,Business & Licensing,Education & Training,Health Services,Identity & Civil Affairs,Justice & Notary,Labour & HR Services,Municipal Services,Social Support & Subsidies,Traffic & Transport
region,,,,,,,,,
Al-Baha,1.9,2.9,1.6,2.2,2.4,2.4,2.3,2.3,2.1
Al-Jouf,8.2,8.0,8.7,8.9,8.6,8.3,8.4,8.4,8.7
Asir,2.2,2.4,2.1,2.1,2.6,2.6,2.3,2.4,2.0
Eastern Province,7.0,7.7,7.7,8.3,7.9,7.5,7.9,7.8,8.2
Hail,8.2,8.6,7.8,7.7,7.7,7.7,7.7,8.0,8.5
Jazan,2.3,2.4,2.1,3.1,3.2,3.1,2.5,2.3,2.8
Madinah,8.6,8.1,8.3,8.1,7.9,8.6,8.1,7.9,7.8
Makkah,7.3,8.3,8.2,7.0,7.3,8.0,8.9,7.8,8.0
Najran,2.0,1.6,2.6,2.7,2.0,2.4,1.9,2.3,2.1


**Result:** inside the five regions, **every** service category gained ≤ 3.2 pp. Everywhere else,
**every** category gained ≥ 7.0 pp. The two ranges do not overlap, so the slowdown appears across
all service categories rather than being driven by one or two weak categories.

**(c) Are these regions too small to matter?**

In [11]:
last12 = df[df["month"] > PREV]
share = last12.groupby("region")["transactions"].sum()
share_pct = 100 * share / share.sum()

print(f"Five slowed regions handled {share_pct[SLOWED].sum():.1f}% of all national transactions "
      f"in the 12 months to Jun 2026 ({share[SLOWED].sum():,} transactions).")
share_pct.round(1).sort_values(ascending=False).to_frame("share of transactions (%)")

Five slowed regions handled 15.6% of all national transactions in the 12 months to Jun 2026 (23,092,866 transactions).


,share of transactions (%)
region,
Makkah,26.5
Riyadh,25.3
Eastern Province,15.1
Madinah,6.6
Asir,6.5
Jazan,4.8
Qassim,4.2
Tabuk,2.9
Hail,2.1


**Result:** the five regions accounted for **15.6% of national transaction volume** in the 12 months to June 2026, so they represent a substantial share of service activity.

## 8. Chart style

One style applied to both charts, so the pair reads as a single report.
Following the course's design principles: colour is spent only on the signal, everything else is grey;
gridlines are faint; the title carries the takeaway; there is no legend, because direct labels
remove the eye's round trip.

In [12]:
ORANGE = "#D55E00"
GREY   = "#C4C4C4"
INK    = "#1A1A1A"
MUTED  = "#6B6B6B"
FONT   = "Helvetica Neue, Helvetica, Arial, sans-serif"

Path("charts").mkdir(exist_ok=True)

def style(fig, title, subtitle="", height=560, margin=None):
    sub = f"<br><span style='font-size:13px;color:{MUTED}'>{subtitle}</span>" if subtitle else ""
    fig.update_layout(
        title=dict(text=f"<b>{title}</b>{sub}",
                   x=0, xanchor="left", y=0.95, font=dict(size=19, color=INK)),
        font=dict(family=FONT),
        plot_bgcolor="white", paper_bgcolor="white",
        showlegend=False, height=height,
        margin=margin or dict(l=150, r=80, t=130, b=95))
    return fig

## 9. Chart 1 — change over time

**Question:** Has growth slowed in some regions but not others?

**Chart type:** Line chart, because the question is about change over time. The five slowed regions are highlighted in orange while the other eight are grey, reducing clutter from 13 regional lines. Direct labels replace the legend, and the shaded area marks the Jun 2024–Jun 2026 decision window. The y-axis is truncated because a line chart represents change through position and slope rather than bar length.

In [13]:
fig1 = go.Figure()
for r in region.columns:
    hit = r in SLOWED
    fig1.add_trace(go.Scatter(
        x=region.index, y=region[r], mode="lines",
        line=dict(color=ORANGE if hit else GREY, width=2.6 if hit else 1.3),
        hovertemplate=f"<b>{r}</b><br>%{{x|%b %Y}}: %{{y:.1f}}%<extra></extra>"))

# The decision window, shaded and labelled in one call
fig1.add_vrect(x0=MID, x1=LAST, fillcolor="#F4F4F4", layer="below", line_width=0,
               annotation_text="Decision window", annotation_position="top",
               annotation_font=dict(size=12, color=MUTED))

for group, colour, label in [(SLOWED, ORANGE, "5 slowed regions"),
                             (gain.drop(SLOWED).index, "#8A8A8A", "Other 8 regions")]:
    fig1.add_annotation(x=LAST, y=region.loc[LAST, group].max(), xshift=10,
                        text=f"<b>{label}</b>", showarrow=False, xanchor="left",
                        font=dict(size=12, color=colour))


fig1.update_xaxes(title=dict(text="Month", font=dict(size=13, color=MUTED)),
                  showgrid=False, tickfont=dict(size=12, color=MUTED),
                  range=["2021-05-01", "2026-12-15"])
fig1.update_yaxes(title=dict(text="Digital adoption (% of transactions)",
                             font=dict(size=13, color=MUTED)),
                  range=[25, 74], showgrid=True, gridcolor="#F0F0F0",
                  tickfont=dict(size=12, color=MUTED))

style(fig1,
      "Five regions slowed sharply after mid-2024 while the other eight kept growing",
      margin=dict(l=90, r=160, t=110, b=70))

fig1.show(config={"toImageButtonOptions": {
    "format": "png", "filename": "chart1_adoption_trend",
    "width": 1000, "height": 560, "scale": 2
}})

**Takeaway:** the five orange regions were growing alongside the rest until mid-2024, then their growth slowed sharply. They are not the lowest lines on the chart — Madinah starts lower and later overtakes them — which is why ranking regions by their adoption level would miss this slowdown.

## 10. Chart 2 — comparison between groups

**Question:** Which regions gained least over the decision window?

**Chart type:** Sorted horizontal bar, because the question is about ranking regions by their two-year gain. The zero baseline keeps the bar lengths honest, truncating a bar axis would exaggerate the differences, while direct labels show the exact values and the dashed national line shows the national pace.

In [14]:
g = gain.sort_values(ascending=False)

fig2 = go.Figure(go.Bar(
    x=g.values, y=g.index, orientation="h", width=0.68,
    marker=dict(color=[ORANGE if r in SLOWED else GREY for r in g.index]),
    text=[f"+{v:.1f}" for v in g.values], textposition="outside",
    textfont=dict(size=13, color=INK),
    hovertemplate="<b>%{y}</b><br>%{x:.2f} percentage points gained<extra></extra>"))

# National reference line
fig2.add_vline(x=nat_growth, line_dash="dash", line_color=MUTED, line_width=1.2)
fig2.add_annotation(x=nat_growth, y=-0.6, xshift=6, xanchor="left", showarrow=False,
                    text=f"National +{nat_growth:.1f} percentage points", font=dict(size=12, color=MUTED))

fig2.update_xaxes(title=dict(text="Gain in digital adoption (percentage points)",
                             font=dict(size=13, color=MUTED)),
                  range=[0, g.max() * 1.18], showgrid=True, gridcolor="#F0F0F0",
                  tickfont=dict(size=12, color=MUTED))
fig2.update_yaxes(showgrid=False, tickfont=dict(size=13, color=INK))

style(
    fig2,
    "Five regions gained only 2.2–2.6 percentage points, while the other eight gained 7.7–8.5 percentage points",
    "<b>Jun 2024 to Jun 2026</b>",
    margin=dict(l=160, r=90, t=120, b=80)
)

fig2.show(config={"toImageButtonOptions": {
    "format": "png", "filename": "chart2_region_gain",
    "width": 1000, "height": 560, "scale": 2}})

**Takeaway:** the other eight regions gained about **3.5 times as many percentage points on average** as the five slowed regions, and no region sits between the two groups.